Gold Layer --> Generating KPI Tables

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# defining spark session
spark = SparkSession.builder.appName('JobETL_Gold').getOrCreate()

# reading silver layer
print("✅Reading from Silver Layer -------> Golden Layer")
gold_df = spark.table('job_market.silver.jobs')

#writing the golden layer to a file 
gold_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable("job_market.gold.jobs")

# KPI 1 - Top Companies by job postings
top_companies = (
    gold_df.groupBy('company_name')
    .agg(
        count('*').alias('job_count')
    )
    .orderBy(col('job_count').desc())
    .limit(5)
)
# display(top_companies)

# KPI 2 - Total job postings by city
total_city_jobs = (
    gold_df.groupBy('location')
    .agg(
        count('*').alias('job_count')
    )
    .orderBy(col('job_count').desc())
)
# display(total_city_jobs)

In [0]:
df=spark.table('job_market.silver.jobs')
df.write.mode("append").saveAsTable("JobAggregatorTable")
print("✅Data Successfully Written to Table ---> JobAggregatorTable")

In [0]:
print("😍Congratulations!!!!! Full ETL Job Aggregator Project is Completed and LIVE!!!!")

In [0]:
%sql 
describe history JobAggregatorTable

In [0]:
%sql
select * from JobAggregatorTable

## Data Analytics Part :- Visualising Business Scenrios

In [0]:
import matplotlib.pyplot as plt

In [0]:
spark = SparkSession.builder.appName("VisualsFromGold").getOrCreate()

In [0]:
# Load gold layer: top companies
df_companies = spark.table('job_market.gold.jobs')
df_companies = df_companies.dropna(subset=["posted_at"])
display(df_companies)

In [0]:
# Convert to Pandas for plotting
pdf = df_companies.orderBy("posted_at", ascending=True).limit(10).toPandas()

In [0]:
display(pdf)

In [0]:
#creating a plot
plt.figure(figsize=(10,5))
plt.barh(pdf['company_name'][::-1], pdf['title'][::-1], color='skyblue')
plt.xlabel('Number of Jobs')
plt.ylabel('Top Companies Hiring')
plt.grid(True,axis='x',linestyle='--',alpha=0.5)
plt.tight_layout()
plt.show()

## Job Demand by City Plot

In [0]:
df_cities = spark.table('job_market.gold.jobs')
df_cities=df_cities.dropna(subset=["posted_at"])
pdf_city = df_cities.orderBy("posted_at", ascending=False).toPandas()

In [0]:
# Plot
plt.figure(figsize=(10, 6))
plt.barh(pdf_city['location'][::-1], pdf_city['title'][::-1], color='lightgreen')
plt.xlabel("Job Postings")
plt.title("Job Demand by City")
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()